In [ ]:
# ============================================================
# Cell 0 — Setup: repo + imports (same template style as yours)
# ============================================================
import os, sys, math, gc, copy
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- clone repo if needed and add to path
if not Path("vit-dyt").exists():
    !git clone https://github.com/labofdoubt/vit-dyt
sys.path.insert(0, str(Path("vit-dyt").resolve()))

from timm.models import create_model
from datasets import build_dataset
from dynamic_tanh import convert_ln_to_derf, DynamicErf

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

def seed_all(seed: int = 0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

seed_all(0)

In [ ]:
# ============================================================
# REPLACE Cell 1 — Knobs (BATCH_SIZE can stay here as a default)
# ============================================================
MODEL_NAME  = "vit_base_patch16_224"
NUM_CLASSES = 100
IMG_SIZE    = 224

# Default batch size; you can override it per run now.
BATCH_SIZE_DEFAULT = 8

# Key requirement: make depth variable (passed to create_model via build_vit)
DEPTH = 24  # change this freely

# DERF alpha sweep defaults (you can override in the run function too)
ALPHAS_DEFAULT = np.arange(0.1, 2.0 + 1e-9, 0.2).astype(float)

# Match your original file’s GELU -> ReLU patch switches
REPLACE_GELU_WITH_RELU = True
INPLACE_RELU = False

criterion = nn.CrossEntropyLoss()
print("DEPTH =", DEPTH, "| BATCH_SIZE_DEFAULT =", BATCH_SIZE_DEFAULT)

In [ ]:
# ============================================================
# REPLACE Cell 2 — Load CIFAR100 once; sample a fixed batch per run
# ============================================================
def get_cifar100_train_dataset(img_size: int):
    """
    Uses vit-dyt's build_dataset() like your notebook.
    With nb_classes=100 and data_set="CIFAR", this corresponds to CIFAR-100 in that codebase.
    """
    args = SimpleNamespace(
        data_set="CIFAR",
        data_path="tmp/cifar",
        eval_data_path=None,
        nb_classes=NUM_CLASSES,
        input_size=img_size,
        imagenet_default_mean_and_std=True,
        color_jitter=0.4,
        aa="rand-m9-mstd0.5-inc1",
        train_interpolation="bicubic",
        reprob=0.25,
        remode="pixel",
        recount=1,
        crop_pct=None,
    )
    train_dataset, _ = build_dataset(is_train=True, args=args)
    return train_dataset

TRAIN_DATASET = get_cifar100_train_dataset(IMG_SIZE)

def sample_fixed_batch_from_dataset(train_dataset, batch_size: int, *, seed: int = 0):
    seed_all(seed)
    g = torch.Generator()
    g.manual_seed(seed)

    loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=g,
        num_workers=0,
        pin_memory=False,
        drop_last=True,
    )
    samples, targets = next(iter(loader))
    return samples, targets

# quick smoke test
_s, _t = sample_fixed_batch_from_dataset(TRAIN_DATASET, BATCH_SIZE_DEFAULT, seed=0)
print("Sample batch:", _s.shape, _t.shape)
del _s, _t

In [ ]:
# ============================================================
# NEW Cell — MLP weight scaling utility
# ============================================================
def scale_vit_mlp_init_std(model: nn.Module, multiplier: float):
    """
    Multiply the weights of each ViT block's MLP fc1/fc2 layers by `multiplier`.
    """
    if multiplier <= 0:
        raise ValueError("MLP init std multiplier must be positive.")
    if not hasattr(model, "blocks"):
        raise ValueError("MLP init std scaling currently supports ViT models with a `blocks` attribute.")

    with torch.no_grad():
        for block in model.blocks:
            mlp = getattr(block, "mlp", None)
            if mlp is None:
                continue
            for attr in ("fc1", "fc2"):
                layer = getattr(mlp, attr, None)
                if isinstance(layer, nn.Linear):
                    layer.weight.mul_(multiplier)

In [ ]:
# ============================================================
# Cell 3 — ViT build helpers (GELU->ReLU + depth override)
# ============================================================
def replace_gelu_with_relu(module: nn.Module, *, inplace_relu: bool = False) -> nn.Module:
    """Recursively replace nn.GELU with nn.ReLU in-place (same idea as your notebook)."""
    for name, child in module.named_children():
        if isinstance(child, nn.GELU):
            setattr(module, name, nn.ReLU(inplace=inplace_relu))
        else:
            replace_gelu_with_relu(child, inplace_relu=inplace_relu)
    return module

def build_vit(model_name: str, *, num_classes: int, use_derf: bool, depth: int):
    """
    Requirement:
      1) Load ViT via timm.create_model like your code
      2) Replace GELU->ReLU like your code
      3) Allow varying depth via depth=... in create_model
      4) If use_derf=True, convert LN -> DERF
    """
    model = create_model(
        model_name,
        pretrained=False,
        num_classes=num_classes,
        global_pool="avg",
        drop_path_rate=0.0,
        depth=int(depth),   # <-- requirement #2
    )

    if REPLACE_GELU_WITH_RELU:
        replace_gelu_with_relu(model, inplace_relu=INPLACE_RELU)

    if use_derf:
        model = convert_ln_to_derf(model)

    model.eval().to(DEVICE)
    return model

@torch.no_grad()
def set_all_derf_alpha_(model: nn.Module, alpha_value: float) -> int:
    """Set alpha on all DynamicErf modules (same style as your notebook)."""
    n = 0
    a = float(alpha_value)
    for m in model.modules():
        if isinstance(m, DynamicErf):
            m.alpha_init_value = a
            m.alpha.data.fill_(a)
            n += 1
    return n

In [ ]:
# ============================================================
# REPLACE Cell 4 — Memory-safe gradient capture + exclude_cls
#   - stores only scalar energies per block (floats), not full tensors
# ============================================================
def _mean_sq_norm_over_batch_and_seq(g: torch.Tensor, *, exclude_cls: bool) -> torch.Tensor:
    """
    g: [B, N, D]
    Returns scalar tensor: mean_{b,n} ||g[b,n,:]||^2
    Optionally excludes CLS token (index 0 along N).
    """
    if exclude_cls:
        if g.shape[1] <= 1:
            raise ValueError("exclude_cls=True but sequence length N<=1.")
        g = g[:, 1:, :]
    return g.pow(2).sum(dim=-1).mean(dim=(0, 1))

def compute_grad_energy_profile_one_batch(
    model: nn.Module,
    images: torch.Tensor,
    labels: torch.Tensor,
    *,
    exclude_cls: bool = False,
) -> np.ndarray:
    """
    One forward/backward on a fixed batch.

    Returns G of length (L+1):
      G[0]   = E||grad wrt input to block0||^2  (special point)
      G[i+1] = E||grad wrt output of block i||^2, i=0..L-1

    IMPORTANT: We store only scalar energies (floats) computed inside hooks
    to avoid GPU OOM when depth is large / repeated runs.
    """
    assert hasattr(model, "blocks"), "Expected a timm ViT with model.blocks"
    L = len(model.blocks)

    # storage for scalar energies
    e_in0 = {"val": None}                 # scalar energy for input to first block
    e_out = [None] * L                    # scalar energies for each block output
    x0_ref = {"x0": None}                 # leaf tensor to read x0.grad
    handles = []

    # capture input to block0 as a leaf so we can read x0.grad
    def pre_hook_block0(mod, inputs):
        x_in = inputs[0]  # [B,N,D]
        # no clone -> less memory; still safe because we return this tensor downstream
        x0 = x_in.detach().requires_grad_(True)
        x0_ref["x0"] = x0
        return (x0,)

    # compute scalar energy of grad_output for each block inside backward hook
    def make_block_bwd_hook(i: int):
        def _hook(mod, grad_input, grad_output):
            go = grad_output[0] if isinstance(grad_output, (tuple, list)) else grad_output
            with torch.no_grad():
                e = _mean_sq_norm_over_batch_and_seq(go, exclude_cls=exclude_cls)
            e_out[i] = float(e.detach().cpu().item())
        return _hook

    handles.append(model.blocks[0].register_forward_pre_hook(pre_hook_block0))
    for i in range(L):
        handles.append(model.blocks[i].register_full_backward_hook(make_block_bwd_hook(i)))

    try:
        model.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()

        x0 = x0_ref["x0"]
        if x0 is None or x0.grad is None:
            raise RuntimeError("Failed to capture x0.grad (input to block 0).")
        if any(v is None for v in e_out):
            raise RuntimeError("Failed to capture some block output gradient energies.")

        with torch.no_grad():
            e0 = _mean_sq_norm_over_batch_and_seq(x0.grad, exclude_cls=exclude_cls)

        G = np.zeros(L + 1, dtype=np.float64)
        G[0] = float(e0.detach().cpu().item())
        for i in range(L):
            G[i + 1] = e_out[i]
        return G

    finally:
        for h in handles:
            h.remove()
        # make sure we drop refs promptly
        x0_ref["x0"] = None

In [ ]:
# ============================================================
# REPLACE Cell 5 — Gradient amplification relative to LAST block
#   amp[k] = E||g_k||^2 / E||g_L||^2  => last point is always 1
# ============================================================
def compute_amplification_to_last(G: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    G length = L+1:
      G[0]   : input-to-block0 grad energy
      G[1:]  : block output grad energies (block 0 .. block L-1)
      G[-1]  : output grad energy of last block

    Returns AMP length L+1:
      AMP[k] = G[k] / G[-1], so AMP[-1] == 1
    """
    G = np.asarray(G, dtype=np.float64)
    denom = G[-1] + eps
    return G / denom

In [ ]:
# ============================================================
# REPLACE run_grad_amplification_experiment — add MLP_MULT
# ============================================================
def run_grad_amplification_experiment(
    *,
    depth: int,
    alphas: np.ndarray = ALPHAS_DEFAULT,
    model_name: str = MODEL_NAME,
    num_classes: int = NUM_CLASSES,
    img_size: int = IMG_SIZE,
    batch_size: int = BATCH_SIZE_DEFAULT,
    data_seed: int = 0,
    model_seed: int = 0,
    exclude_cls: bool = False,
    train_dataset=None,
    MLP_MULT: float = 1.0,     # <-- NEW
):
    """
    Returns:
      preln: {G, amp}
      derf:  list of {alpha, G, amp}
      meta

    MLP_MULT:
      Scales ViT MLP (fc1/fc2) weights by this multiplier (applied on CPU once),
      then the same scaled weights are used for both pre-LN and DERF models.
    """
    if train_dataset is None:
        train_dataset = TRAIN_DATASET

    # fixed batch for THIS run (same across pre-LN and all alphas)
    samples_cpu, targets_cpu = sample_fixed_batch_from_dataset(
        train_dataset, batch_size=batch_size, seed=data_seed
    )

    images = samples_cpu.to(DEVICE, dtype=torch.float32, non_blocking=False)
    labels = targets_cpu.to(DEVICE, dtype=torch.long, non_blocking=False)

    preln_model = None
    derf_model = None

    try:
        # ---- build pre-LN model on CPU (deterministic init), keep CPU state_dict
        seed_all(model_seed)
        preln_model = build_vit(model_name, num_classes=num_classes, use_derf=False, depth=depth)

        # build_vit currently puts model on DEVICE; move to CPU first
        preln_model_cpu = preln_model.to("cpu").eval()

        # ---- apply MLP scaling ON CPU (once), before saving base weights
        if MLP_MULT is None:
            MLP_MULT = 1.0
        MLP_MULT = float(MLP_MULT)
        if MLP_MULT != 1.0:
            scale_vit_mlp_init_std(preln_model_cpu, MLP_MULT)

        # save scaled weights
        base_sd = {k: v.detach().clone() for k, v in preln_model_cpu.state_dict().items()}  # CPU tensors

        # ---- run pre-LN on GPU
        preln_model = preln_model_cpu.to(DEVICE).eval()
        G_pre = compute_grad_energy_profile_one_batch(preln_model, images, labels, exclude_cls=exclude_cls)
        amp_pre = compute_amplification_to_last(G_pre)

        # free pre-LN model before creating DERF
        del preln_model
        cuda_cleanup()

        # ---- build DERF model on CPU, load SAME (scaled) weights, then convert LN->DERF on CPU
        derf_model = create_model(
            model_name,
            pretrained=False,
            num_classes=num_classes,
            global_pool="avg",
            drop_path_rate=0.0,
            depth=int(depth),
        )
        if REPLACE_GELU_WITH_RELU:
            replace_gelu_with_relu(derf_model, inplace_relu=INPLACE_RELU)

        derf_model.load_state_dict(base_sd, strict=True)
        derf_model = convert_ln_to_derf(derf_model).eval()  # CPU
        derf_model = derf_model.to(DEVICE)

        # ---- DERF sweep
        alphas = np.asarray(alphas, dtype=float)
        derf_results = []
        for a in alphas:
            set_all_derf_alpha_(derf_model, float(a))
            G = compute_grad_energy_profile_one_batch(derf_model, images, labels, exclude_cls=exclude_cls)
            amp = compute_amplification_to_last(G)
            derf_results.append({"alpha": float(a), "G": G, "amp": amp})

            derf_model.zero_grad(set_to_none=True)
            cuda_cleanup()

        out = {
            "preln": {"G": G_pre, "amp": amp_pre},
            "derf": derf_results,
            "meta": {
                "model_name": model_name,
                "depth": int(depth),
                "num_classes": int(num_classes),
                "batch_size": int(batch_size),
                "img_size": int(img_size),
                "exclude_cls": bool(exclude_cls),
                "gelu_to_relu": bool(REPLACE_GELU_WITH_RELU),
                "mlp_mult": float(MLP_MULT),
            },
        }
        return out

    finally:
        # ensure we drop EVERYTHING even if an exception happens mid-run
        try:
            if preln_model is not None:
                del preln_model
        except Exception:
            pass
        try:
            if derf_model is not None:
                del derf_model
        except Exception:
            pass
        try:
            del images, labels, samples_cpu, targets_cpu
        except Exception:
            pass
        cuda_cleanup()

In [ ]:
!apt-get -qq update
!apt-get -qq install texlive-latex-extra texlive-fonts-recommended dvipng cm-super

In [ ]:
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")

In [ ]:
# ============================================================
# NeurIPS-like LaTeX fonts in Matplotlib WITHOUT neurips_2025.sty
# ============================================================
# !apt-get -qq update
# !apt-get -qq install texlive-latex-extra texlive-fonts-recommended dvipng cm-super

import shutil
from pathlib import Path
import matplotlib as mpl
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats("svg")

# Clear TeX cache (important after changing preamble)
tex_cache = Path(mpl.get_cachedir()) / "tex.cache"
if tex_cache.exists():
    shutil.rmtree(tex_cache)

NEURIPS_LIKE_PREAMBLE = r"""
\usepackage[T1]{fontenc}
\usepackage{newtxtext}
\usepackage{newtxmath}

\usepackage{microtype}
\usepackage{xcolor}

\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage{bm}
\usepackage{accents}

\usepackage{nicefrac}
\usepackage{xfrac}

% your macros (optional)
\newcommand{\dbtilde}[1]{\accentset{\approx}{#1}}
\newcommand{\vardbtilde}[1]{\tilde{\raisebox{0pt}[0.87\height]{$\tilde{#1}$}}}
"""

mpl.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": NEURIPS_LIKE_PREAMBLE,

    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.2,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
import shutil
from pathlib import Path
import matplotlib as mpl

# Clear cache after changing preamble
tex_cache = Path(mpl.get_cachedir()) / "tex.cache"
if tex_cache.exists():
    shutil.rmtree(tex_cache)

mpl.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"""
\usepackage[T1]{fontenc}
\usepackage{mathptmx}     % Times-like text+math (close to NeurIPS look)
\usepackage{amsmath,amssymb,amsfonts,bm}
\usepackage{microtype}
""",
})

In [ ]:
# ============================================================
# DROP-IN REPLACEMENT — Cell 7 (Plotting, LaTeX+SVG + tunable font handles)
# ============================================================
# Assumes you already ran:
#  - apt-get texlive... cell
#  - mpl.rcParams.update({... "text.usetex": True, ...})
#  - AMS preamble fix for \mathbb (amsmath/amssymb/amsfonts)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import LogLocator, LogFormatterMathtext, NullFormatter

# ---- defaults (override here if you want)
BASE_CMAP_NAME = globals().get("BASE_CMAP_NAME", "viridis")

# colorbar layout (tune if you like)
CVAL_MIN   = globals().get("CVAL_MIN", 0.08)
CVAL_MAX   = globals().get("CVAL_MAX", 0.92)
CVAL_GAMMA = globals().get("CVAL_GAMMA", 0.95)
COLORBAR_WIDTH  = globals().get("COLORBAR_WIDTH", 0.62)
COLORBAR_HEIGHT = globals().get("COLORBAR_HEIGHT", 0.75)


def shade_from_index(i: int, n: int, light=CVAL_MIN, dark=CVAL_MAX, gamma=CVAL_GAMMA) -> float:
    """Map index i in [0..n-1] to a colormap value in [light..dark] with gamma shaping."""
    if n <= 1:
        return 0.5 * (light + dark)
    t = i / (n - 1)
    t = t ** gamma
    return light + (dark - light) * t


def prettify_log_axis(ax, axis="y"):
    locator_major = LogLocator(base=10.0, numticks=10)
    locator_minor = LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=100)
    formatter = LogFormatterMathtext(base=10.0)
    if axis == "y":
        ax.yaxis.set_major_locator(locator_major)
        ax.yaxis.set_minor_locator(locator_minor)
        ax.yaxis.set_major_formatter(formatter)
        ax.yaxis.set_minor_formatter(NullFormatter())
    else:
        ax.xaxis.set_major_locator(locator_major)
        ax.xaxis.set_minor_locator(locator_minor)
        ax.xaxis.set_major_formatter(formatter)
        ax.xaxis.set_minor_formatter(NullFormatter())


def prettify_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(which="both", top=False, right=False)
    ax.grid(True, which="major", linewidth=0.5, alpha=0.22)
    ax.grid(False, which="minor")


def add_alpha_colorbar_horizontal_single(
    cax,
    alphas: np.ndarray,
    colors,
    *,
    label=r"$\alpha$",
    alpha_legend_fs: float = 8,
    max_tick_labels: int = 7,
):
    """
    Draw a compact horizontal colorbar-like strip mapping alpha->color,
    with ticks at a subset of alpha values.

    alpha_legend_fs controls BOTH:
      - alpha label fontsize
      - alpha tick label fontsize
    """
    alphas = np.asarray(alphas, dtype=float)
    n = len(alphas)

    img = np.zeros((1, n, 4))
    for i in range(n):
        img[0, i, :] = colors[i]

    da = float(np.mean(np.diff(alphas))) if n > 1 else 1.0
    x0, x1 = float(alphas[0] - da / 2), float(alphas[-1] + da / 2)

    cax.imshow(img, origin="lower", aspect="auto", extent=(x0, x1, 0, 1))
    cax.set_ylim(0, 1)
    cax.set_yticks([])

    if n <= max_tick_labels:
        idx = np.arange(n)
    else:
        idx = np.unique(np.linspace(0, n - 1, num=max_tick_labels, dtype=int))

    tick_vals = alphas[idx]
    cax.set_xticks(tick_vals)
    cax.set_xticklabels([f"{v:g}" for v in tick_vals], fontsize=alpha_legend_fs)
    cax.tick_params(
        axis="x",
        which="both",
        bottom=True,
        labelbottom=True,
        top=False,
        labeltop=False,
        length=0,
        width=0,
        pad=2,
    )

    cax.set_frame_on(False)
    for sp in cax.spines.values():
        sp.set_visible(False)

    cax.set_xlabel(label, fontsize=alpha_legend_fs)
    cax.xaxis.set_label_position("top")
    return cax


def plot_amplification_coeffs(
    result: dict,
    *,
    title: str | None = None,
    logy: bool = True,
    drop_first_point: bool = False,
    figsize=(6.0, 6.0),

    # --------- NEW TUNABLE HANDLES (as requested) ----------
    tick_label_fs: float | None = None,     # 1) tick labels
    axis_label_fs: float | None = None,     # 2) axis labels
    title_fs: float | None = None,          # 3) title
    alpha_legend_fs: float | None = None,   # 4) alpha label + alpha tick labels (colorbar strip)
    preln_fs: float | None = None,          # 5) "pre-LN" label fontsize (legend)
):
    """
    Plots gradient amplification:
      amp[k] = E||g_k||^2 / E||g_L||^2

    Indexing / x-axis:
      k=0  -> special point: grad w.r.t. input to first block (x=-1)
      k=1..L -> grads at outputs of blocks 0..L-1 (x=0..L-1)
      last plotted point (x=L-1) is always 1.

    Styling:
      DERF curves use viridis; pre-LN is black.
      Font sizes are tunable via the handles above.
    """
    # ---- default font sizes fall back to current rcParams if not provided
    if tick_label_fs is None:
        tick_label_fs = float(mpl.rcParams.get("xtick.labelsize", 8))
    if axis_label_fs is None:
        axis_label_fs = float(mpl.rcParams.get("axes.labelsize", 9))
    if title_fs is None:
        title_fs = float(mpl.rcParams.get("axes.titlesize", mpl.rcParams.get("font.size", 9)))
    if alpha_legend_fs is None:
        alpha_legend_fs = float(mpl.rcParams.get("legend.fontsize", 8))
    if preln_fs is None:
        preln_fs = float(mpl.rcParams.get("legend.fontsize", 8))

    derf = result["derf"]
    pre_amp = np.asarray(result["preln"]["amp"], dtype=float)  # length L+1
    Lp1 = pre_amp.shape[0]
    L = Lp1 - 1

    # x: [-1, 0, 1, ..., L-1] aligns with amp indices [0..L]
    x_full = np.concatenate((np.array([-1], dtype=int), np.arange(L, dtype=int)))

    if drop_first_point:
        x = x_full[1:]
        pre_plot = pre_amp[1:]
    else:
        x = x_full
        pre_plot = pre_amp

    alphas = np.array([r["alpha"] for r in derf], dtype=float)
    base_cmap = plt.get_cmap(BASE_CMAP_NAME)
    cvals  = [shade_from_index(i, len(alphas)) for i in range(len(alphas))]
    colors = [base_cmap(cv) for cv in cvals]

    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1.0, 0.22], hspace=0.45)
    ax = fig.add_subplot(gs[0, 0])
    cax = fig.add_subplot(gs[1, 0])

    # center/shrink the colorbar axis
    pos = cax.get_position()
    new_w = pos.width * COLORBAR_WIDTH
    new_h = pos.height * COLORBAR_HEIGHT
    new_x = pos.x0 + (pos.width - new_w) / 2
    new_y = pos.y0 + (pos.height - new_h) / 2
    cax.set_position([new_x, new_y, new_w, new_h])

    # DERF curves
    for i, r in enumerate(derf):
        amp = np.asarray(r["amp"], dtype=float)
        amp_plot = amp[1:] if drop_first_point else amp
        ax.plot(x, amp_plot, color=colors[i])

    # pre-LN curve
    ax.plot(x, pre_plot, color="black", linewidth=1.6, label=r"pre-LN")

    # labels / ticks / title
    ax.set_xlabel(r"layer index $l$", fontsize=axis_label_fs)
    ax.set_ylabel(r"$\mathbb{E}\|g_l\|^2 \,/\, \mathbb{E}\|g_L\|^2$", fontsize=axis_label_fs)
    ax.set_xlim(int(x.min()), int(x.max()))
    ax.tick_params(axis="both", which="both", labelsize=tick_label_fs)

    if logy:
        ax.set_yscale("log")
        prettify_log_axis(ax, "y")
        ax.tick_params(axis="y", which="both", labelsize=tick_label_fs)

    prettify_axes(ax)

    if title is None:
        meta = result.get("meta", {})
        depth = meta.get("depth", L)
        title = rf"Gradient amplification (depth={depth})"
        if meta.get("exclude_cls", False):
            title += r", excl.\ CLS"
        if meta.get("mlp_mult", 1.0) != 1.0:
            title += rf", MLP\_MULT={meta['mlp_mult']:g}"
    ax.set_title(title, fontsize=title_fs)


    # Alpha colorbar strip (label + values fontsize controlled together)
    add_alpha_colorbar_horizontal_single(
        cax, alphas, colors,
        label=r"$\alpha$",
        alpha_legend_fs=alpha_legend_fs,
        max_tick_labels=7,
    )
    # Pre-LN legend tile adjacent to colorbar (matches vit_apjn styling)
    rect = mpatches.Rectangle((1.02, 0.05), 0.08, 0.9,
                                transform=cax.transAxes, facecolor="black", edgecolor="black",
                                clip_on=False)
    cax.add_patch(rect)
    cax.text(1.06, -0.2, "pre-LN", transform=cax.transAxes, ha="center", va="top",
             fontsize=preln_fs)

    plt.show()

In [ ]:
# ============================================================
# REPLACE Cell 8 — Example run: now controls batch_size/exclude_cls/drop_first_point
# ============================================================
# Default batch size; you can override it per run now.

MLP_MULT = 4.2
DEPTH = 12  # change this freely
ALPHAS_DEFAULT = np.arange(0.1, 2.0 + 1e-9, 0.2).astype(float)

result = run_grad_amplification_experiment(
    depth=DEPTH,
    alphas=ALPHAS_DEFAULT,
    model_name=MODEL_NAME,
    batch_size=16,        # change freely
    exclude_cls=True,     # switch
    MLP_MULT=MLP_MULT,
    data_seed=0,
    model_seed=0,
)

In [ ]:
plot_amplification_coeffs(
    result,
    logy=True,
    drop_first_point=True,
    tick_label_fs=18,
    axis_label_fs=22,
    title_fs=24,
    alpha_legend_fs=24,
    preln_fs=24,
)

In [ ]:
# ============================================================
# UPDATED Cell — all alpha ticks + piece width, pre-LN legend right of cbar,
# markers for datapoints with size control, panel (b) includes x-ticks 10 and 30
# ============================================================

ALPHAS_DEFAULT = np.arange(0.1, 2.0 + 1e-9, 0.2).astype(float)

def add_alpha_colorbar_horizontal_allticks(
    cax,
    alphas: np.ndarray,
    colors,
    *,
    piece_width: float = 1.0,          # <-- NEW: width of each color tile
    label: str = r"$\alpha$",
    alpha_legend_fs: float = 8,
    tick_pad: float = 3.0,
    tick_length: float = 2.5,
):
    """
    Horizontal alpha->color strip where each alpha gets its own tick label.
    The x-axis is in "tile units" (equal-width pieces), with tick labels = alphas.
    """
    alphas = np.asarray(alphas, dtype=float)
    n = len(alphas)

    img = np.zeros((1, n, 4))
    for i in range(n):
        img[0, i, :] = colors[i]

    # Equal-width tiles; axis in [0, n*piece_width]
    x0, x1 = 0.0, float(n) * float(piece_width)
    cax.imshow(img, origin="lower", aspect="auto", extent=(x0, x1, 0, 1))
    cax.set_ylim(0, 1)
    cax.set_yticks([])

    # Tick at center of each tile, label with actual alpha value
    centers = (np.arange(n, dtype=float) + 0.5) * float(piece_width)
    cax.set_xticks(centers)
    cax.set_xticklabels([f"{a:g}" for a in alphas], fontsize=alpha_legend_fs)

    cax.tick_params(
        axis="x",
        which="both",
        bottom=True,
        labelbottom=True,
        top=False,
        labeltop=False,
        length=tick_length,
        width=0.8,
        pad=tick_pad,
    )

    cax.set_xlim(x0, x1)
    cax.set_xlabel(label, fontsize=alpha_legend_fs)

    cax.set_frame_on(False)
    for sp in cax.spines.values():
        sp.set_visible(False)

    return cax


def plot_amplification_triptych(
    results: list[dict],
    *,
    titles: list[str],
    alphas: np.ndarray,
    logy: bool = True,
    drop_first_point: bool = True,
    figsize=(12.5, 3.8),

    # font handles
    tick_label_fs: float | None = None,
    axis_label_fs: float | None = None,
    title_fs: float | None = None,
    alpha_legend_fs: float | None = None,
    preln_fs: float | None = None,

    # spacing controls (kept)
    wspace: float = 0.28,
    hspace: float = 0.60,
    legend_y: float = 1.2,  # kept for API compatibility; legend is now placed next to cbar

    # NEW params
    marker_size: float = 3.5,           # datapoint marker size (points)
    curve_linewidth: float = 1.6,       # NEW: line width for all curves
    colorbar_piece_width: float = 1.0,  # width per alpha tile in the shared bar
    cbar_legend_pad_x: float = 0.015,   # gap between colorbar and "pre-LN" label (figure coords)
):
    if tick_label_fs is None:
        tick_label_fs = float(mpl.rcParams.get("xtick.labelsize", 8))
    if axis_label_fs is None:
        axis_label_fs = float(mpl.rcParams.get("axes.labelsize", 9))
    if title_fs is None:
        title_fs = float(mpl.rcParams.get("axes.titlesize", mpl.rcParams.get("font.size", 9)))
    if alpha_legend_fs is None:
        alpha_legend_fs = float(mpl.rcParams.get("legend.fontsize", 8))
    if preln_fs is None:
        preln_fs = float(mpl.rcParams.get("legend.fontsize", 8))

    alphas = np.asarray(alphas, dtype=float)

    # shared alpha->color mapping
    base_cmap = plt.get_cmap(BASE_CMAP_NAME)
    cvals  = [shade_from_index(i, len(alphas)) for i in range(len(alphas))]
    colors = [base_cmap(cv) for cv in cvals]

    # figure grid: 1x3 plots + shared colorbar row
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(
        nrows=2, ncols=3,
        height_ratios=[1.0, 0.22],
        hspace=hspace, wspace=wspace
    )
    axs = [fig.add_subplot(gs[0, i]) for i in range(3)]
    cax = fig.add_subplot(gs[1, :])

    # center/shrink shared colorbar axis (uses your existing globals)
    pos = cax.get_position()
    new_w = pos.width * COLORBAR_WIDTH
    new_h = pos.height * COLORBAR_HEIGHT
    new_x = pos.x0 + (pos.width - new_w) / 2
    new_y = pos.y0 + (pos.height - new_h) / 2
    cax.set_position([new_x, new_y, new_w, new_h])

    preln_lines = []
    derf_lines  = [[] for _ in range(3)]
    title_texts = []

    for j, (ax, res) in enumerate(zip(axs, results)):
        derf = res["derf"]
        pre_amp = np.asarray(res["preln"]["amp"], dtype=float)
        L = pre_amp.shape[0] - 1

        x_full = np.concatenate((np.array([-1], dtype=int), np.arange(L, dtype=int)))
        if drop_first_point:
            x = x_full[1:]
            pre_plot = pre_amp[1:]
        else:
            x = x_full
            pre_plot = pre_amp

        # DERF curves with markers
        for i, r in enumerate(derf):
            amp = np.asarray(r["amp"], dtype=float)
            amp_plot = amp[1:] if drop_first_point else amp
            ln, = ax.plot(
                x, amp_plot,
                color=colors[i],
                linewidth=curve_linewidth,
                marker="o",
                markersize=marker_size,
                markerfacecolor=colors[i],
                markeredgecolor=colors[i],
                markeredgewidth=0.0,
            )
            derf_lines[j].append(ln)

        # pre-LN curve with markers too (black)
        pre_ln, = ax.plot(
            x, pre_plot,
            color="black",
            linewidth=curve_linewidth,
            marker="o",
            markersize=marker_size,
            markerfacecolor="black",
            markeredgecolor="black",
            markeredgewidth=0.0,
            label=r"pre-LN",
        )
        preln_lines.append(pre_ln)

        # cosmetics
        ax.set_xlabel(r"$l$", fontsize=axis_label_fs)
        ax.tick_params(axis="both", which="both", labelsize=tick_label_fs)

        if logy:
            ax.set_yscale("log")
            prettify_log_axis(ax, "y")

            # third panel: reduce y-label density to every other decade
            if j == 2:
                ymin, ymax = ax.get_ylim()
                if ymin > 0 and ymax > 0:
                    emin = int(np.floor(np.log10(ymin)))
                    emax = int(np.ceil(np.log10(ymax)))

                    odd_power_ticks = [
                        10.0 ** e for e in range(emin, emax + 1)
                        if e % 2 != 0
                    ]

                    if odd_power_ticks:
                        ax.yaxis.set_major_locator(mpl.ticker.FixedLocator(odd_power_ticks))
                        ax.yaxis.set_major_formatter(mpl.ticker.LogFormatterMathtext(base=10))
                        ax.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

        prettify_axes(ax)

        # titles
        tt = ax.set_title(titles[j], fontsize=title_fs, pad=8)
        title_texts.append(tt)

        # y-label only on left-most; BUT y ticks/tickmarks on all three
        if j == 0:
            ax.set_ylabel(
                r"$\mathbb{E}\lVert g_{l} \rVert^2 \,/\, \mathbb{E}\lVert g_{L}\rVert^2$",
                fontsize=axis_label_fs
            )
        else:
            ax.set_ylabel("")

        # panel (b): ensure x-ticks include 10 and 30
        if j == 1:
            cur = list(ax.get_xticks())
            cur.extend([10.0, 30.0])
            xmin, xmax = ax.get_xlim()
            cur = sorted({t for t in cur if xmin - 1e-9 <= t <= xmax + 1e-9})
            ax.set_xticks(cur)

    # shared alpha strip: SHOW ALL TICKS + controllable tile width
    add_alpha_colorbar_horizontal_allticks(
        cax,
        alphas,
        colors,
        piece_width=colorbar_piece_width,
        label=r"$\alpha$ (Derf)",
        alpha_legend_fs=alpha_legend_fs,
    )

    # "pre-LN" legend rendered as a black tile adjacent to the shared colorbar
    cpos = cax.get_position()
    axes_pad = cbar_legend_pad_x / max(cpos.width, 1e-9)
    rect_x = 1.0 + axes_pad
    rect = mpatches.Rectangle((rect_x, 0.05), 0.08, 0.9,
                                transform=cax.transAxes, facecolor="black", edgecolor="black",
                                clip_on=False)
    cax.add_patch(rect)
    text_x = rect_x + 0.04
    cax.text(text_x, -0.2, "pre-LN", transform=cax.transAxes, ha="center", va="top",
             fontsize=preln_fs)
    preln_legend = rect

    handles = {
        "axs": axs,
        "cax": cax,
        "preln_lines": preln_lines,
        "derf_lines": derf_lines,
        "title_texts": title_texts,
        "preln_legend": preln_legend,
    }
    return fig, axs, cax, handles

In [ ]:
!apt-get -qq update
!apt-get -qq install texlive-latex-extra texlive-fonts-recommended dvipng cm-super

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../00-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package fonts-lato.
Preparing to unpack .../01-fonts-lato_2.0-2.1_all.deb ...
Unpacking fonts-lato (2.0-2.1) ...
Selecting previously unselected package poppler-data.
Preparing to unpack .../02-poppler-data_0.4.11-1_all.deb ...
Unpacking poppler-data (0.4.11-1) ...
Selecting previously unselected package tex-common.
Preparing to unpack .../03-tex-common_6.17_all.deb ...
Unpacking tex-common (6.17) ...
Selecting previously unselect

In [ ]:
# -------------------------
# Run the three experiments
# -------------------------
EXCLUDE_CLS = True
BATCH_SIZE  = BATCH_SIZE_DEFAULT
DATA_SEED   = 1
MODEL_SEED  = 1

res_a = run_grad_amplification_experiment(
    depth=12,
    alphas=ALPHAS_DEFAULT,
    batch_size=BATCH_SIZE,
    exclude_cls=EXCLUDE_CLS,
    data_seed=DATA_SEED,
    model_seed=MODEL_SEED,
    MLP_MULT=1.0,
)

res_b = run_grad_amplification_experiment(
    depth=48,
    alphas=ALPHAS_DEFAULT,
    batch_size=BATCH_SIZE,
    exclude_cls=EXCLUDE_CLS,
    data_seed=DATA_SEED,
    model_seed=MODEL_SEED,
    MLP_MULT=1.0,
)

res_c = run_grad_amplification_experiment(
    depth=12,
    alphas=ALPHAS_DEFAULT,
    batch_size=BATCH_SIZE,
    exclude_cls=EXCLUDE_CLS,
    data_seed=DATA_SEED,
    model_seed=MODEL_SEED,
    MLP_MULT=4.2,
)

In [ ]:

# -------------------------
# Plot (a)(b)(c) + shared alpha bar
# -------------------------
titles = [
    r"(a) Depth 12",
    r"(b) Depth 48",
    r"(c) Depth 12; MLP std $\times 4$",
]

fig, axs, cax, handles = plot_amplification_triptych(
    [res_a, res_b, res_c],
    titles=titles,
    alphas=ALPHAS_DEFAULT,
    logy=True,
    drop_first_point=True,

    # font handles
    tick_label_fs=18,
    axis_label_fs=20,
    title_fs=20,
    alpha_legend_fs=20,
    preln_fs=18,

    # spacing controls (tweak these two if you want even more gap)
    hspace=0.70,
    legend_y=1.2,
    curve_linewidth=1.0,
    marker_size=1.8,
)
plt.show()

In [ ]:
# higher precision text/lines in many viewers
fig.savefig("grad_amplification_triptych_.pdf", bbox_inches="tight", dpi=300)

**Tiles: depth vs. alpha and sigma mult vs. alpha**

In [ ]:
!apt-get -qq update
!apt-get -qq install texlive-latex-extra texlive-fonts-recommended dvipng cm-super

In [ ]:
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")

In [ ]:
# ============================================================
# NeurIPS-like LaTeX fonts in Matplotlib WITHOUT neurips_2025.sty
# ============================================================
# !apt-get -qq update
# !apt-get -qq install texlive-latex-extra texlive-fonts-recommended dvipng cm-super

import shutil
from pathlib import Path
import matplotlib as mpl
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats("svg")

# Clear TeX cache (important after changing preamble)
tex_cache = Path(mpl.get_cachedir()) / "tex.cache"
if tex_cache.exists():
    shutil.rmtree(tex_cache)

NEURIPS_LIKE_PREAMBLE = r"""
\usepackage[T1]{fontenc}
\usepackage{newtxtext}
\usepackage{newtxmath}

\usepackage{microtype}
\usepackage{xcolor}

\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage{bm}
\usepackage{accents}

\usepackage{nicefrac}
\usepackage{xfrac}

% your macros (optional)
\newcommand{\dbtilde}[1]{\accentset{\approx}{#1}}
\newcommand{\vardbtilde}[1]{\tilde{\raisebox{0pt}[0.87\height]{$\tilde{#1}$}}}
"""

mpl.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": NEURIPS_LIKE_PREAMBLE,

    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.2,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
import shutil
from pathlib import Path
import matplotlib as mpl

# Clear cache after changing preamble
tex_cache = Path(mpl.get_cachedir()) / "tex.cache"
if tex_cache.exists():
    shutil.rmtree(tex_cache)

mpl.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"""
\usepackage[T1]{fontenc}
\usepackage{mathptmx}     % Times-like text+math (close to NeurIPS look)
\usepackage{amsmath,amssymb,amsfonts,bm}
\usepackage{microtype}
""",
})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================
# Cell 1 — imports + helpers
# =========================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable


def read_log_jsonl(output_dir: Path, filename: str = "log.txt") -> pd.DataFrame:
    log_path = output_dir / filename
    if not log_path.exists():
        raise FileNotFoundError(log_path)

    rows = []
    with log_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue

    if not rows:
        raise ValueError(f"No valid JSON rows in {log_path}")

    df = pd.DataFrame(rows)
    if "epoch" in df.columns:
        df = df.sort_values("epoch").reset_index(drop=True)
    return df


def get_last_value(df: pd.DataFrame, col: str, target_epoch: int | None = None) -> float:
    """
    Prefer the row with epoch == target_epoch (or last epoch <= target_epoch),
    otherwise fall back to the last row.
    """
    if col not in df.columns:
        raise KeyError(f"missing column: {col}")

    if "epoch" in df.columns and target_epoch is not None:
        df2 = df[df["epoch"] <= target_epoch].copy()
        if len(df2) == 0:
            df2 = df.copy()
    else:
        df2 = df

    return float(df2[col].iloc[-1])


def _masked_minmax(*dfs: pd.DataFrame) -> tuple[float, float]:
    arrs = []
    for df in dfs:
        a = df.to_numpy(dtype=float).reshape(-1)
        a = a[~np.isnan(a)]
        if len(a):
            arrs.append(a)
    if not arrs:
        return 0.0, 1.0
    allv = np.concatenate(arrs, axis=0)
    return float(allv.min()), float(allv.max())


def _plot_annotated_imshow(
    ax,
    df: pd.DataFrame,
    *,
    x_labels,
    y_labels,
    xlabel: str,
    ylabel: str,
    title: str,
    fmt: str,
    cmap,
    norm,
    tick_label_fs: float,
    axis_label_fs: float,
    title_fs: float,
    annot_fs: float,
    show_y_ticklabels: bool = True,
):
    data = np.ma.masked_invalid(df.to_numpy(dtype=float))
    im = ax.imshow(data, aspect="auto", interpolation="nearest", cmap=cmap, norm=norm)

    ax.set_xticks(np.arange(len(x_labels)))
    ax.set_xticklabels([str(x) for x in x_labels], fontsize=tick_label_fs)

    ax.set_yticks(np.arange(len(y_labels)))
    if show_y_ticklabels:
        ax.set_yticklabels([str(y) for y in y_labels], fontsize=tick_label_fs)
    else:
        ax.set_yticklabels([])

    ax.set_xlabel(xlabel, fontsize=axis_label_fs)
    ax.set_ylabel(ylabel, fontsize=axis_label_fs)
    ax.set_title(title, fontsize=title_fs, pad=8)

    # annotate cells
    for i, y in enumerate(y_labels):
        for j, x in enumerate(x_labels):
            v = df.loc[y, x]
            txt = "—" if pd.isna(v) else format(float(v), fmt)
            ax.text(j, i, txt, ha="center", va="center", fontsize=annot_fs)

    ax.invert_yaxis()
    return im

In [ ]:
# ==========================================
# Cell 2 — experiment configs + dir templates
# ==========================================

# (a) depth vs alpha (DERF) + pre-LN tile (depth only)
ROOT_DEPTH = Path("/content/drive/MyDrive/ml_projects/norm_free_transformer_depth")
EPOCHS_DEPTH = 20
WARMUP_EPOCHS_DEPTH = 3
LR_DEPTH = 3e-4

DEPTHS = [12, 18, 24, 30, 36]
ALPHAS_DEPTH = [0.4, 0.5, 0.6, 0.7, 0.8]

def make_output_dir_derf_depth(root: Path, depth: int, epochs: int, warmup_epochs: int, alpha: float, lr: float) -> Path:
    return root / f"derf_depth_{depth}_epochs_{epochs}_warmup_{warmup_epochs}_alpha_{alpha}_lr_{lr}_no_lr_decay"

def make_output_dir_preln_depth(root: Path, depth: int, epochs: int, warmup_epochs: int, lr: float) -> Path:
    return root / f"pre_ln_depth_{depth}_epochs_{epochs}_warmup_{warmup_epochs}_lr_{lr}_no_lr_decay"


# (b) mult vs alpha (DERF) + pre-LN tile (mult only) with DEPTH fixed
ROOT_SIGMA = Path("/content/drive/MyDrive/ml_projects/norm_free_transformer_sigma")
DEPTH_FIXED = 12
EPOCHS_SIGMA = 15
WARMUP_EPOCHS_SIGMA = 3
LR_SIGMA = 3e-4

MULTS = [1.0, 1.8, 2.6, 3.4, 4.2]
ALPHAS_SIGMA = [0.4, 0.5, 0.6, 0.7]

def make_output_dir_derf_mult(root: Path, mult: float, depth: int, epochs: int, warmup_epochs: int, alpha: float, lr: float) -> Path:
    return root / (
        f"derf_mult_{mult}_depth_{depth}_epochs_{epochs}_warmup_{warmup_epochs}"
        f"_alpha_{alpha}_lr_{lr}_no_lr_decay"
    )

def make_output_dir_preln_mult(root: Path, mult: float, depth: int, epochs: int, warmup_epochs: int, lr: float) -> Path:
    return root / (
        f"pre_ln_mult_{mult}_depth_{depth}_epochs_{epochs}_warmup_{warmup_epochs}"
        f"_lr_{lr}_no_lr_decay"
    )

In [ ]:
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import numpy as np
import matplotlib.pyplot as plt

def cmap_from_base_clip_whiten(
    base_name="RdYlGn",
    *,
    clip_lo=0.08,
    clip_hi=0.92,
    whiten=0.0,
    N=256,
    name=None,
):
    """
    Take an existing cmap (e.g. RdYlGn), clip its darkest ends, and optionally mix with white.
    This keeps the 'RdYlGn' look but removes very dark red/green.
    """
    base = plt.get_cmap(base_name)
    xs = np.linspace(clip_lo, clip_hi, N)
    cols = base(xs)  # RGBA
    if whiten > 0:
        cols[:, :3] = cols[:, :3] * (1 - whiten) + 1.0 * whiten
    if name is None:
        name = f"{base_name}_clip{clip_lo:.2f}-{clip_hi:.2f}_w{whiten:.2f}"
    return ListedColormap(cols, name=name)

def make_rdylgn_clean_yellow(name="RdYlGn_clean_yellow", N=256):
    """
    Hand-tuned R-Y-G with a clean yellow mid and lighter ends.
    Designed to avoid dark endpoints and muddy orange.
    """
    stops = [
        (0.00, "#ef3b2c"),  # bright red (not dark)
        (0.22, "#fb6a4a"),  # salmon (avoids dirty brown)
        (0.50, "#ffeb3b"),  # clean yellow
        (0.78, "#a1d99b"),  # light green
        (1.00, "#31a354"),  # medium green (not dark)
    ]
    cmap = LinearSegmentedColormap.from_list(name, stops, N=N)
    return cmap

# --- Add/extend your CMAPS dict ---
# If CMAPS already exists, this will update it; otherwise it will create it.
try:
    CMAPS
except NameError:
    CMAPS = {}

# 1) Very close to RdYlGn, just less dark at ends
CMAPS["RdYlGn_soft"] = cmap_from_base_clip_whiten("RdYlGn", clip_lo=0.08, clip_hi=0.92, whiten=0.00, name="RdYlGn_soft")

# 2) Same, but slightly cleaner/lighter overall (often best for papers)
CMAPS["RdYlGn_soft_pastel"] = cmap_from_base_clip_whiten("RdYlGn", clip_lo=0.08, clip_hi=0.92, whiten=0.10, name="RdYlGn_soft_pastel")

# 3) Custom “clean yellow” version (closest to your stated taste)
CMAPS["RdYlGn_clean_yellow"] = make_rdylgn_clean_yellow()

# Make sure NaNs look consistent
for k in ["RdYlGn_soft", "RdYlGn_soft_pastel", "RdYlGn_clean_yellow"]:
    CMAPS[k] = CMAPS[k].copy()
    CMAPS[k].set_bad(color="lightgray")

In [ ]:
# ============================================================
# Cell 3 — collectors + ONE figure with (a) and (b) + shared cbar
# Changes:
#  1) extra horizontal gap between (a) and (b): ab_gap
#  2) extra vertical gap between tiles and colorbar row: legend_gap
#  3) shift (a)/(b) titles to the right: ab_title_xshift (axes fraction)
# ============================================================

def collect_depth_alpha_metric(metric: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    target_epoch = EPOCHS_DEPTH - 1
    derf_df = pd.DataFrame(index=DEPTHS, columns=ALPHAS_DEPTH, dtype=float)
    pre_df  = pd.DataFrame(index=DEPTHS, columns=["pre-LN"], dtype=float)

    for depth in DEPTHS:
        for alpha in ALPHAS_DEPTH:
            out_dir = make_output_dir_derf_depth(ROOT_DEPTH, depth, EPOCHS_DEPTH, WARMUP_EPOCHS_DEPTH, alpha, LR_DEPTH)
            try:
                df = read_log_jsonl(out_dir)
                derf_df.loc[depth, alpha] = get_last_value(df, metric, target_epoch=target_epoch)
            except Exception as e:
                print(f"[SKIP][DERF-depth] depth={depth}, alpha={alpha} -> {out_dir.name} ({e})")

    for depth in DEPTHS:
        out_dir = make_output_dir_preln_depth(ROOT_DEPTH, depth, EPOCHS_DEPTH, WARMUP_EPOCHS_DEPTH, LR_DEPTH)
        try:
            df = read_log_jsonl(out_dir)
            pre_df.loc[depth, "pre-LN"] = get_last_value(df, metric, target_epoch=target_epoch)
        except Exception as e:
            print(f"[SKIP][preLN-depth] depth={depth} -> {out_dir.name} ({e})")

    return derf_df, pre_df


def collect_mult_alpha_metric(metric: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    target_epoch = EPOCHS_SIGMA - 1
    derf_df = pd.DataFrame(index=MULTS, columns=ALPHAS_SIGMA, dtype=float)
    pre_df  = pd.DataFrame(index=MULTS, columns=["pre-LN"], dtype=float)

    for mult in MULTS:
        for alpha in ALPHAS_SIGMA:
            out_dir = make_output_dir_derf_mult(ROOT_SIGMA, mult, DEPTH_FIXED, EPOCHS_SIGMA, WARMUP_EPOCHS_SIGMA, alpha, LR_SIGMA)
            try:
                df = read_log_jsonl(out_dir)
                derf_df.loc[mult, alpha] = get_last_value(df, metric, target_epoch=target_epoch)
            except Exception as e:
                print(f"[SKIP][DERF-mult] mult={mult}, alpha={alpha} -> {out_dir.name} ({e})")

    for mult in MULTS:
        out_dir = make_output_dir_preln_mult(ROOT_SIGMA, mult, DEPTH_FIXED, EPOCHS_SIGMA, WARMUP_EPOCHS_SIGMA, LR_SIGMA)
        try:
            df = read_log_jsonl(out_dir)
            pre_df.loc[mult, "pre-LN"] = get_last_value(df, metric, target_epoch=target_epoch)
        except Exception as e:
            print(f"[SKIP][preLN-mult] mult={mult} -> {out_dir.name} ({e})")

    return derf_df, pre_df


def plot_combined_depth_and_mult_tiles(
    *,
    split: str = "test",  # "test" or "train"
    # style params (match your earlier convention)
    tick_label_fs: float = 18,
    axis_label_fs: float = 22,
    title_fs: float = 22,
    alpha_legend_fs: float = 20,
    annot_fs: float = 14,
    # layout (existing)
    wspace: float = 0.25,
    hspace: float = 0.35,
    cbar_height: float = 0.035,

    # --- NEW HANDLES ---
    ab_gap: float = 0.60,          # (1) extra horizontal gap between (a) and (b) groups (in "pre-LN tile widths")
    legend_gap: float = 0.30,      # (2) extra vertical gap between top row and colorbar row (additive to hspace)
    ab_title_xshift: float = 0.08, # (3) shift (a)/(b) title to the right (axes fraction, 0=no shift)
):
    split = str(split).strip().lower()
    if split not in {"test", "train"}:
        raise ValueError("split must be 'test' or 'train'")

    if split == "test":
        metric = "test_acc1"
        fmt = ".2f"
        cmap_name = "RdYlGn_soft_pastel"
        cbar_label = "Test accuracy"
    else:
        metric = "train_loss"
        fmt = ".3f"
        cmap_name = "RdYlGn_r"
        cbar_label = "train_loss"

    derf_depth, pre_depth = collect_depth_alpha_metric(metric)
    derf_mult,  pre_mult  = collect_mult_alpha_metric(metric)

    vmin, vmax = _masked_minmax(derf_depth, pre_depth, derf_mult, pre_mult)
    norm = Normalize(vmin=vmin, vmax=vmax)

    # prettier green↔red
    cmap = (CMAPS[cmap_name] if split == "test" else CMAPS["brewer_soft"].reversed()).copy()
    cmap.set_bad(color="lightgray")

    # cmap = plt.get_cmap(cmap_name).copy()
    # cmap.set_bad(color="lightgray")

    fig_w = max(14.0, 0.75 * (len(ALPHAS_DEPTH) + len(ALPHAS_SIGMA)) + 8.0)
    fig_h = max(5.2, 0.50 * max(len(DEPTHS), len(MULTS)) + 3.2)

    fig = plt.figure(figsize=(fig_w, fig_h))

    # --- (1) add a dedicated spacer column between (a) and (b) ---
    # col0: a-derf, col1: a-pre, col2: spacer, col3: b-derf, col4: b-pre
    # spacer width scales by ab_gap (in units of "pre tile width")
    gs = fig.add_gridspec(
        nrows=2, ncols=5,
        height_ratios=[1.0, 0.16],
        width_ratios=[
            max(1, len(ALPHAS_DEPTH)),
            1,
            float(ab_gap),
            max(1, len(ALPHAS_SIGMA)),
            1,
        ],
        wspace=wspace,
        hspace=hspace + legend_gap,  # (2) more space above colorbar row
    )

    ax_a_derf = fig.add_subplot(gs[0, 0])
    ax_a_pre  = fig.add_subplot(gs[0, 1])
    ax_b_derf = fig.add_subplot(gs[0, 3])
    ax_b_pre  = fig.add_subplot(gs[0, 4])
    cax       = fig.add_subplot(gs[1, :])
    cax.set_axis_off()

    # (a)
    _plot_annotated_imshow(
        ax_a_derf, derf_depth,
        x_labels=ALPHAS_DEPTH, y_labels=DEPTHS,
        xlabel=r"$\alpha$ (Derf)", ylabel="depth",
        title=r"(a) Depth sweep (20 epochs)",
        fmt=fmt, cmap=cmap, norm=norm,
        tick_label_fs=tick_label_fs, axis_label_fs=axis_label_fs,
        title_fs=title_fs, annot_fs=annot_fs,
        show_y_ticklabels=True,
    )
    _plot_annotated_imshow(
        ax_a_pre, pre_depth,
        x_labels=["pre-LN"], y_labels=DEPTHS,
        xlabel="", ylabel="",
        title="",
        fmt=fmt, cmap=cmap, norm=norm,
        tick_label_fs=tick_label_fs, axis_label_fs=axis_label_fs,
        title_fs=title_fs, annot_fs=annot_fs,
        show_y_ticklabels=False,
    )

    # (b)
    _plot_annotated_imshow(
        ax_b_derf, derf_mult,
        x_labels=ALPHAS_SIGMA, y_labels=MULTS,
        xlabel=r"$\alpha$ (Derf)", ylabel="MLP std scale",
        title=r"(b) MLP std scale sweep (15 epochs)",
        fmt=fmt, cmap=cmap, norm=norm,
        tick_label_fs=tick_label_fs, axis_label_fs=axis_label_fs,
        title_fs=title_fs, annot_fs=annot_fs,
        show_y_ticklabels=True,
    )
    _plot_annotated_imshow(
        ax_b_pre, pre_mult,
        x_labels=["pre-LN"], y_labels=MULTS,
        xlabel="", ylabel="",
        title="",
        fmt=fmt, cmap=cmap, norm=norm,
        tick_label_fs=tick_label_fs, axis_label_fs=axis_label_fs,
        title_fs=title_fs, annot_fs=annot_fs,
        show_y_ticklabels=False,
    )

    # --- (3) shift titles (a) and (b) to the right ---
    # We shift via set_title(..., x=...) where x is in axes fraction
    # Default is x=0.5 (center). So we use 0.5 + ab_title_xshift.
    ax_a_derf.set_title(ax_a_derf.get_title(), fontsize=title_fs, pad=8, x=0.5 + ab_title_xshift)
    ax_b_derf.set_title(ax_b_derf.get_title(), fontsize=title_fs, pad=8, x=0.5 + ab_title_xshift)

    # shared colorbar below
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cpos = cax.get_position()
    bar_ax = fig.add_axes([
        cpos.x0 + 0.12 * cpos.width,
        cpos.y0 + 0.35 * cpos.height,
        0.76 * cpos.width,
        cbar_height,
    ])
    cbar = fig.colorbar(sm, cax=bar_ax, orientation="horizontal")
    cbar.ax.tick_params(labelsize=alpha_legend_fs)
    cbar.set_label(cbar_label, fontsize=axis_label_fs)

    handles = {
        "ax_a_derf": ax_a_derf,
        "ax_a_pre": ax_a_pre,
        "ax_b_derf": ax_b_derf,
        "ax_b_pre": ax_b_pre,
        "cbar": cbar,
        "bar_ax": bar_ax,

        # expose new layout handles
        "ab_gap": ab_gap,
        "legend_gap": legend_gap,
        "ab_title_xshift": ab_title_xshift,
    }
    return fig, (ax_a_derf, ax_a_pre, ax_b_derf, ax_b_pre), cbar, handles, (derf_depth, pre_depth, derf_mult, pre_mult)


# --- Create the combined figure (default: test) ---
fig, axes, cbar, handles, dfs = plot_combined_depth_and_mult_tiles(
    split="test",  # change to "train" for train_loss
    tick_label_fs=18,
    axis_label_fs=22,
    title_fs=22,
    alpha_legend_fs=20,
    annot_fs=14,

    # NEW knobs
    ab_gap=0.10,          # (1) bigger => more horizontal space between (a) and (b)
    legend_gap=0.1,      # (2) bigger => more vertical space above colorbar row
    ab_title_xshift=0.20, # (3) bigger => titles move right
)
plt.show()

In [ ]:
fig.savefig(
    "depth_and_mlp_scale_tiles.pdf",
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
)

**Derf warmup and lr sweeps**

In [ ]:
# =========================
# Cell 1 — imports + helpers
# =========================
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable


def read_log_jsonl(output_dir: Path, filename: str = "log.txt") -> pd.DataFrame:
    log_path = output_dir / filename
    if not log_path.exists():
        raise FileNotFoundError(log_path)

    rows = []
    with log_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue

    if not rows:
        raise ValueError(f"No valid JSON rows in {log_path}")

    df = pd.DataFrame(rows)
    if "epoch" in df.columns:
        df = df.sort_values("epoch").reset_index(drop=True)
    return df


def get_last_value_first_k_epochs(df: pd.DataFrame, col: str, k: int, *, target_epoch: int | None = None) -> float:
    """
    Take value from the last row within the first k epochs (epoch 0..k-1) if epoch column exists,
    otherwise first k rows. If target_epoch is provided, prefer <= target_epoch (still within first k).
    """
    if col not in df.columns:
        raise KeyError(f"missing column: {col}")

    if "epoch" in df.columns:
        d = df[df["epoch"] < k].copy()
        if target_epoch is not None:
            d = d[d["epoch"] <= target_epoch].copy()
    else:
        d = df.iloc[:k].copy()

    if len(d) == 0:
        raise ValueError("no rows available for requested epoch window")

    return float(d[col].iloc[-1])


def _masked_minmax(*dfs: pd.DataFrame) -> tuple[float, float]:
    arrs = []
    for df in dfs:
        a = df.to_numpy(dtype=float).reshape(-1)
        a = a[~np.isnan(a)]
        if len(a):
            arrs.append(a)
    if not arrs:
        return 0.0, 1.0
    allv = np.concatenate(arrs, axis=0)
    return float(allv.min()), float(allv.max())


def _get_cmap(name_or_key: str, reverse: bool = False):
    """
    If a CMAPS dict exists in the notebook, use it; else fall back to plt.get_cmap.
    """
    cmap = None
    if "CMAPS" in globals() and isinstance(globals().get("CMAPS"), dict) and name_or_key in CMAPS:
        cmap = CMAPS[name_or_key]
    else:
        cmap = plt.get_cmap(name_or_key)

    cmap = cmap.reversed() if reverse else cmap
    cmap = cmap.copy()
    cmap.set_bad(color="lightgray")
    return cmap


def _plot_annotated_imshow(
    ax,
    df: pd.DataFrame,
    *,
    x_labels,
    y_labels,
    xlabel: str,
    ylabel: str,
    title: str,
    fmt: str,
    cmap,
    norm,
    tick_label_fs: float,
    axis_label_fs: float,
    title_fs: float,
    annot_fs: float,
):
    data = np.ma.masked_invalid(df.to_numpy(dtype=float))
    im = ax.imshow(data, aspect="auto", interpolation="nearest", cmap=cmap, norm=norm)

    ax.set_xticks(np.arange(len(x_labels)))
    ax.set_xticklabels([str(x) for x in x_labels], fontsize=tick_label_fs)

    ax.set_yticks(np.arange(len(y_labels)))
    ax.set_yticklabels([str(y) for y in y_labels], fontsize=tick_label_fs)

    ax.set_xlabel(xlabel, fontsize=axis_label_fs)
    ax.set_ylabel(ylabel, fontsize=axis_label_fs)
    ax.set_title(title, fontsize=title_fs, pad=8)

    for i, y in enumerate(y_labels):
        for j, x in enumerate(x_labels):
            v = df.loc[y, x]
            txt = "—" if pd.isna(v) else format(float(v), fmt)
            ax.text(j, i, txt, ha="center", va="center", fontsize=annot_fs)

    ax.invert_yaxis()
    return im


def _make_output_dir_simple(root: Path, epochs: int, warmup_epochs: int, alpha: float, lr: float) -> Path:
    # Matches your warmup/LR sweep naming scheme
    return root / f"derf_epochs_{epochs}_warmup_{warmup_epochs}_alpha_{alpha}_lr_{lr}_no_lr_decay"


def _make_output_dir_smallalpha_robust(
    root: Path,
    *,
    epochs: int,
    warmup_epochs: int,
    lr: float,
    dynamic_tanh: bool,
    dynamic_erf: bool,
    alpha: float,
) -> Path:
    """
    Tries a few common naming patterns; if none exist, falls back to a glob search.
    This makes the (c) panel robust to slight naming differences.
    """
    lr_s = f"{lr}"
    a_s = f"{alpha}"

    candidates = [
        # simplest (same as warmup/LR sweeps)
        root / f"derf_epochs_{epochs}_warmup_{warmup_epochs}_alpha_{a_s}_lr_{lr_s}_no_lr_decay",

        # with dynamic flags
        root / (
            f"derf_epochs_{epochs}_warmup_{warmup_epochs}_alpha_{a_s}_lr_{lr_s}"
            f"_dynamic_tanh_{dynamic_tanh}_dynamic_erf_{dynamic_erf}_no_lr_decay"
        ),

        # alternative token for alpha
        root / (
            f"derf_epochs_{epochs}_warmup_{warmup_epochs}_lr_{lr_s}"
            f"_dynamic_tanh_{dynamic_tanh}_dynamic_erf_{dynamic_erf}"
            f"_dyt_alpha_init_value_{a_s}_no_lr_decay"
        ),
    ]
    for p in candidates:
        if p.exists():
            return p

    # fallback: glob search
    pats = [
        f"*epochs_{epochs}*warmup_{warmup_epochs}*lr_{lr_s}*alpha*{a_s}*",
        f"*epochs_{epochs}*warmup_{warmup_epochs}*lr_{lr_s}*{a_s}*",
    ]
    hits = []
    for pat in pats:
        hits.extend(sorted(root.glob(pat)))
    hits = [h for h in hits if h.is_dir()]
    if len(hits) == 0:
        # return the "canonical" candidate even if missing (caller will handle)
        return candidates[0]
    return hits[0]

In [ ]:
# ==========================================
# Cell 2 — configs for (a) (b) (c)
# ==========================================

# --- shared root for (a)(b)(c) ---
ROOT_MAIN = Path("/content/drive/MyDrive/ml_projects/norm_free_transformer")

# (a) Warmup sweep
LR_WARMUP = 5e-4
EPOCHS_DEFAULT = 10
EPOCHS_SPECIAL = 30
WARMUP_EPOCHS_LS = [0, 1, 3]
ALPHAS_AB = [0.3, 0.5, 0.7, 0.9]
SPECIAL_WARMUP = 0
SPECIAL_ALPHAS = {0.5, 0.7, 1.0}  # per your note (1.0 may or may not be in ALPHAS_AB)

# (b) LR sweep
WARMUP_FIXED_FOR_LR = 1
LR_LS = [1e-4, 3e-4, 1e-3]

# (c) Small-alpha curves
EPOCHS_CURVES = 90
DYNAMIC_TANH = False
DYNAMIC_ERF = True
LR_CURVES = 3e-4
WARMUP_CURVES = 10
ALPHAS_SMALL = [0.1, 0.2, 0.3, 0.5]

In [ ]:
# ============================================================
# Cell 3 — one figure: (a) warmup sweep, (b) LR sweep, (c) small-alpha curves
# split="test" (default) or "train"
# Added: cbar_gap (vertical distance to colorbar)
# NEW: ab_gap (horizontal distance between (a) and (b))
# Also: removed 0-padding in LR tick labels (1e-4 not 1e-04)
# ============================================================

def _fmt_sci_no_pad(x: float) -> str:
    s = f"{x:.0e}"  # e.g. '1e-04'
    return s.replace("e-0", "e-").replace("e+0", "e+")


def plot_derf_warmup_lr_smallalpha_row(
    *,
    split: str = "test",  # "test" or "train" (default: "test")

    # fonts (same knobs as your other figures)
    tick_label_fs: float = 18,
    axis_label_fs: float = 22,
    title_fs: float = 22,
    alpha_legend_fs: float = 20,
    annot_fs: float = 14,
    legend_fs: float = 16,

    # layout
    figsize=(16.0, 4.8),
    wspace: float = 0.35,
    hspace: float = 0.40,
    cbar_height: float = 0.06,

    # NEW: extra horizontal gap between (a) and (b) (as a fraction of subplot width)
    # 0.0 = no extra gap; typical range 0.03–0.12
    ab_gap: float = 0.06,

    # vertical gap between plots row and colorbar row (in figure coords)
    cbar_gap: float = 0.00,

    # colormap choice (use either a CMAPS key if you defined CMAPS, or a matplotlib cmap name)
    cmap_key: str = "RdYlGn",

    # behavior knobs
    first_k_epochs: int = 10,          # take metric at epoch 9 by default
    special_first_k_epochs: int = 10,  # explicit for clarity
):
    split = str(split).strip().lower()
    if split not in {"test", "train"}:
        raise ValueError("split must be 'test' or 'train'")

    # metric + formatting
    if split == "test":
        metric = "test_acc1"
        fmt_hm = ".2f"
        y_curve = "test_acc1"
        curve_ylabel = "test accuracy"
        cmap = _get_cmap(cmap_key, reverse=False)
        cbar_label = "Test accuracy (10 epochs)"
    else:
        metric = "train_loss"
        fmt_hm = ".3f"
        y_curve = "train_loss"
        curve_ylabel = "train loss"
        cmap = _get_cmap(cmap_key, reverse=True)
        cbar_label = "Train loss at last epoch in run"

    # -------------------------
    # (a) warmup sweep dataframe
    # -------------------------
    warmup_df = pd.DataFrame(index=WARMUP_EPOCHS_LS, columns=ALPHAS_AB, dtype=float)
    for warmup in WARMUP_EPOCHS_LS:
        for alpha in ALPHAS_AB:
            epochs_dir = EPOCHS_DEFAULT
            if (warmup == SPECIAL_WARMUP) and (alpha in SPECIAL_ALPHAS):
                epochs_dir = EPOCHS_SPECIAL

            out_dir = _make_output_dir_simple(ROOT_MAIN, epochs_dir, warmup, alpha, LR_WARMUP)
            try:
                df = read_log_jsonl(out_dir)
                target_epoch = (special_first_k_epochs - 1) if epochs_dir == EPOCHS_SPECIAL else (first_k_epochs - 1)
                warmup_df.loc[warmup, alpha] = get_last_value_first_k_epochs(
                    df, metric, k=10, target_epoch=target_epoch
                )
            except Exception as e:
                print(f"[SKIP][warmup] warmup={warmup}, alpha={alpha} -> {out_dir.name} ({e})")

    # force-fill missing point warmup=0, alpha=0.7 for test
    if split == "test":
        if (0 in warmup_df.index) and (0.7 in warmup_df.columns) and pd.isna(warmup_df.loc[0, 0.7]):
            warmup_df.loc[0, 0.7] = 3.11

    # -------------------------
    # (b) LR sweep dataframe
    # -------------------------
    lr_df = pd.DataFrame(index=LR_LS, columns=ALPHAS_AB, dtype=float)
    for lr in LR_LS:
        for alpha in ALPHAS_AB:
            out_dir = _make_output_dir_simple(ROOT_MAIN, EPOCHS_DEFAULT, WARMUP_FIXED_FOR_LR, alpha, lr)
            try:
                df = read_log_jsonl(out_dir)
                lr_df.loc[lr, alpha] = get_last_value_first_k_epochs(df, metric, k=10, target_epoch=first_k_epochs - 1)
            except Exception as e:
                print(f"[SKIP][lr] lr={lr}, alpha={alpha} -> {out_dir.name} ({e})")

    # shared color scale for heatmaps (a) & (b)
    vmin, vmax = _masked_minmax(warmup_df, lr_df)
    norm = Normalize(vmin=vmin, vmax=vmax)

    # -------------------------
    # (c) curves
    # -------------------------
    curves = {}
    for a in ALPHAS_SMALL:
        out_dir = _make_output_dir_smallalpha_robust(
            ROOT_MAIN,
            epochs=EPOCHS_CURVES,
            warmup_epochs=WARMUP_CURVES,
            lr=LR_CURVES,
            dynamic_tanh=DYNAMIC_TANH,
            dynamic_erf=DYNAMIC_ERF,
            alpha=a,
        )
        try:
            curves[a] = read_log_jsonl(out_dir)
        except Exception as e:
            print(f"[SKIP][curves] alpha={a} -> {out_dir.name} ({e})")

    # -------------------------
    # Plotting
    # -------------------------
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(
        nrows=2, ncols=3,
        height_ratios=[1.0, cbar_height],
        hspace=hspace, wspace=wspace
    )

    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[0, 2])

    cax = fig.add_subplot(gs[1, 0:2])  # colorbar under (a)+(b)
    cax2 = fig.add_subplot(gs[1, 2])
    cax2.set_axis_off()

    # (a)
    _plot_annotated_imshow(
        ax_a, warmup_df,
        x_labels=ALPHAS_AB, y_labels=WARMUP_EPOCHS_LS,
        xlabel=r"$\alpha$", ylabel="warmup epochs",
        title = r"(a) Warmup sweep ($\mathrm{LR}=5\mathrm{e}{-4}$)",
        fmt=fmt_hm, cmap=cmap, norm=norm,
        tick_label_fs=tick_label_fs, axis_label_fs=axis_label_fs,
        title_fs=title_fs, annot_fs=annot_fs,
    )

    # (b) — remove 0 padding in LR labels
    import numpy as np

    def fmt_tex_lr(x: float) -> str:
      exp = int(np.floor(np.log10(x)))
      mant = x / (10**exp)
      if abs(mant - 1.0) < 1e-12:
          return rf"$10^{{{exp}}}$"
      # make mantissa pretty if it’s basically an integer
      if abs(mant - round(mant)) < 1e-12:
          mant = int(round(mant))
      return rf"${mant}\times 10^{{{exp}}}$"

    lr_tick_labels = [fmt_tex_lr(x) for x in LR_LS]

    lr_tick_labels = [_fmt_sci_no_pad(x) for x in LR_LS]   # e.g. 1e-4 instead of 1e-04
    lr_df_for_plot = lr_df.copy()
    lr_df_for_plot.index = lr_tick_labels

    _plot_annotated_imshow(
        ax_b, lr_df_for_plot,
        x_labels=ALPHAS_AB,
        xlabel=r"$\alpha$", ylabel="learning rate",
        y_labels=lr_tick_labels,
        title=r"(b) LR sweep (no warmup)",
        fmt=fmt_hm, cmap=cmap, norm=norm,
        tick_label_fs=tick_label_fs, axis_label_fs=axis_label_fs,
        title_fs=title_fs, annot_fs=annot_fs,
    )

    # (c) curves
    for a, df in curves.items():
        x = df["epoch"] if "epoch" in df.columns else np.arange(len(df), dtype=int)
        if y_curve in df.columns:
            ax_c.plot(x, df[y_curve], label=f"{a:g}")
        else:
            print(f"[SKIP][curves] alpha={a} missing column {y_curve}")

    ax_c.set_xlabel("epoch", fontsize=axis_label_fs)
    ax_c.set_ylabel(curve_ylabel, fontsize=axis_label_fs)
    ax_c.set_title(r"(c) Small-$\alpha$ sweep", fontsize=title_fs, pad=8)
    ax_c.tick_params(axis="both", which="both", labelsize=tick_label_fs)
    ax_c.grid(True, alpha=0.3)
    ax_c.legend(title=r"$\alpha$", frameon=False, fontsize=legend_fs, title_fontsize=legend_fs, loc="best")

    # shared colorbar for (a)(b)
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.ax.tick_params(labelsize=alpha_legend_fs)
    cbar.set_label(cbar_label, fontsize=axis_label_fs)

    # --- adjust vertical position of the colorbar axis (gap control) ---
    if cbar_gap != 0.0:
        pos = cax.get_position()
        cax.set_position([pos.x0, pos.y0 - cbar_gap, pos.width, pos.height])

    # --- NEW: adjust horizontal distance between (a) and (b) only ---
    # We shift (b) and (c) to the right by ab_gap (in figure coordinates) and shrink their widths accordingly,
    # keeping (a) fixed. This avoids changing spacing between (b) and (c).
    if ab_gap != 0.0:
        pa = ax_a.get_position()
        pb = ax_b.get_position()
        pc = ax_c.get_position()
        # shift b and c right by ab_gap; reduce their widths slightly to stay inside the figure
        # (preserves right margin)
        ax_b.set_position([pb.x0 + ab_gap, pb.y0, max(0.01, pb.width - ab_gap * 0.5), pb.height])
        ax_c.set_position([pc.x0 + ab_gap, pc.y0, max(0.01, pc.width - ab_gap * 0.5), pc.height])

        # keep cbar aligned under (a)+(b)
        pbar = cax.get_position()
        cax.set_position([pbar.x0, pbar.y0, max(0.01, pbar.width - ab_gap * 0.5), pbar.height])

    handles = {
        "ax_a": ax_a,
        "ax_b": ax_b,
        "ax_c": ax_c,
        "cax": cax,
        "cbar": cbar,
        "warmup_df": warmup_df,
        "lr_df": lr_df,
        "curves": curves,
        "ab_gap": ab_gap,
    }
    return fig, (ax_a, ax_b, ax_c), handles


# Example:
fig, axs, handles = plot_derf_warmup_lr_smallalpha_row(
    split="test",
    tick_label_fs=18,
    axis_label_fs=22,
    title_fs=22,
    alpha_legend_fs=20,
    annot_fs=14,
    legend_fs=16,
    cmap_key="RdYlGn_soft_pastel",
    cbar_gap=0.05,
    ab_gap=0.005,  # <-- increase to add more horizontal space between (a) and (b)
)
plt.show()

In [ ]:
fig.savefig(
    "warmup_and_lr_tiles.pdf",
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
)